# pMTG Spatial Variant Validation

This notebook defines the bilateral pMTG regions of high inter-individual variability in the cPFM sample (n = 10) and tests how robust they are.

1. **Subject variant maps:** each participant's spatial correlation map is thresholded at its bottom 5%, 10%, and 15% of values, keeping variant clusters of at least 30 mm².
2. **Variant density maps:** binarized variant maps are summed across participants at each threshold.
3. **pMTG regions:** at the 10% threshold, vertices where at least 5 of 10 participants have a variant are clustered (minimum 40 mm²), and the cluster nearest the reported pMTG center is kept in each hemisphere.
4. **Diagnosis exclusion:** pMTG regions are redefined without the three participants with a neurodevelopmental diagnosis (at least 4 of 7) and compared with the full-cohort regions.
5. **Jackknife sensitivity:** pMTG regions are redefined leaving out one participant at a time (at least 4 of 9) and compared with the full-cohort regions and with each other.

Overlap between pMTG regions is measured with the bilateral Dice coefficient.

In [10]:
import subprocess
from itertools import combinations
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

In [ ]:
WORKBENCH = Path("/Applications/workbench/bin_macosx64/wb_command")
VARIANT_DIR = Path(
    "/Users/emk/Documents/Documents - Ron Weasley V/Research/Verbal-Ability/Final/Variants"
)
SURFACE_DIR = Path(
    "/Users/emk/Documents/Documents - Ron Weasley V/Research/Data/Imaging/Surfaces/HCP1200"
)
SURFACE_PATHS = {
    "left": SURFACE_DIR / "S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii",
    "right": SURFACE_DIR / "S1200.R.midthickness_MSMAll.32k_fs_LR.surf.gii",
}

OUTPUT_DIR = Path.cwd() / "variant_spatial_validation_results"
SUBJECT_OUTPUT_DIR = OUTPUT_DIR / "subject_variant_maps"
GROUP_OUTPUT_DIR = OUTPUT_DIR / "group_variant_maps"
SUBJECT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GROUP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# TD = typically developing; ND = neurodevelopmental disorder diagnosis
TD_SUBJECTS = ["MSCPI05", "MSCPI07", "MSCPI08", "MSCPI10", "MSCPI12", "MSCPI17", "MSCPI19"]
ND_SUBJECTS = ["MSCPI14", "MSCPI15", "MSCPI18"]
COHORT_SUBJECTS = sorted(TD_SUBJECTS + ND_SUBJECTS)

VARIANT_PERCENTILES = [5, 10, 15]
SUBJECT_MIN_CLUSTER_AREA_MM2 = 30
GROUP_MIN_CLUSTER_AREA_MM2 = 40  # 30 mm² gives identical pMTG regions

# Reported pMTG centers (MNI mm); the nearest cluster in each hemisphere is the pMTG region
# Used for automatic detection of the pMTG cluster in each hemisphere
PMTG_CENTERS = {
    "left": np.array([-58, -54, 3]),
    "right": np.array([58, -49, 5]),
}

## CIFTI and Workbench Helpers

All maps are cortex-only CIFTI files on the 32k fs_LR surface. Clusters are found with Workbench `-cifti-find-clusters`.

In [12]:
reference_map = nib.load(VARIANT_DIR / f"{COHORT_SUBJECTS[0]}spatialCorrMap.dtseries.nii")
brain_axis = reference_map.header.get_axis(1)

# Surface coordinates of each cortical grayordinate, used to locate cluster centroids
CORTEX_STRUCTURES = {
    "CIFTI_STRUCTURE_CORTEX_LEFT": "left",
    "CIFTI_STRUCTURE_CORTEX_RIGHT": "right",
}
cortex = {}
for structure, structure_slice, brain_model in brain_axis.iter_structures():
    if structure in CORTEX_STRUCTURES:
        hemisphere = CORTEX_STRUCTURES[structure]
        surface_coordinates = nib.load(SURFACE_PATHS[hemisphere]).agg_data("pointset")
        cortex[hemisphere] = {
            "slice": structure_slice,
            "coordinates": surface_coordinates[brain_model.vertex],
        }


def load_cifti_vector(path):
    return np.asarray(nib.load(path).dataobj).squeeze()


def save_dscalar(values, path):
    map_name = path.name.removesuffix(".dscalar.nii")
    header = nib.Cifti2Header.from_axes((nib.cifti2.ScalarAxis([map_name]), brain_axis))
    image = nib.Cifti2Image(np.asarray(values, dtype=np.float32).reshape(1, -1), header=header)
    image.nifti_header.set_intent("ConnDenseScalar")
    nib.save(image, path)


def find_clusters(input_path, threshold, min_area_mm2, output_path, less_than=False):
    """Label surface clusters of values above `threshold` (below it if `less_than`)."""
    command = [
        str(WORKBENCH), "-cifti-find-clusters", str(input_path),
        str(threshold), str(min_area_mm2),
        "0", "0",  # volume threshold and minimum size (unused: maps are cortex-only)
        "COLUMN", str(output_path),
    ]
    if less_than:
        command.append("-less-than")
    command += ["-left-surface", str(SURFACE_PATHS["left"])]
    command += ["-right-surface", str(SURFACE_PATHS["right"])]
    subprocess.run(command, check=True, capture_output=True, text=True)

## Subject Variant Maps

A participant's variants are the vertices in their lowest 5%, 10%, or 15% of spatial correlations with the group average, kept if they form clusters of at least 30 mm².

In [13]:
subject_variant_paths = {}
for percentile in VARIANT_PERCENTILES:
    percentile_dir = SUBJECT_OUTPUT_DIR / f"bottom_{percentile}_percent"
    percentile_dir.mkdir(exist_ok=True)
    subject_variant_paths[percentile] = {}

    for subject in COHORT_SUBJECTS:
        spatial_corr_path = VARIANT_DIR / f"{subject}spatialCorrMap.dtseries.nii"
        cutoff = float(np.percentile(load_cifti_vector(spatial_corr_path), percentile))
        variant_path = (
            percentile_dir
            / f"{subject}_networkVariants_bottom{percentile}_min30mm2.dtseries.nii"
        )
        find_clusters(
            spatial_corr_path,
            cutoff,
            SUBJECT_MIN_CLUSTER_AREA_MM2,
            variant_path,
            less_than=True,
        )
        subject_variant_paths[percentile][subject] = variant_path

## Variant Density Maps

Each density map counts how many participants have a variant at each vertex. These are saved for the full cohort at the 5%, 10%, and 15% thresholds.

In [ ]:
def make_density_map(subjects, percentile, name):
    """Save the number of subjects with a variant at each vertex and return the path."""
    variant_maps = [
        load_cifti_vector(subject_variant_paths[percentile][subject]) != 0
        for subject in subjects
    ]
    density_path = GROUP_OUTPUT_DIR / f"{name}_density.dscalar.nii"
    save_dscalar(np.sum(variant_maps, axis=0), density_path)
    return density_path


full_density_paths = {}
for percentile in VARIANT_PERCENTILES:
    name = f"full_cohort_bottom{percentile}"
    full_density_paths[percentile] = make_density_map(COHORT_SUBJECTS, percentile, name)

## pMTG Regions

At the 10% threshold, vertices where at least 5 of the 10 participants have a variant are clustered (minimum 40 mm²). 

In [15]:
def centroid(mask, hemisphere):
    """Return the mean surface coordinate of the masked vertices in one hemisphere."""
    hemisphere_mask = mask[cortex[hemisphere]["slice"]]
    return cortex[hemisphere]["coordinates"][hemisphere_mask].mean(axis=0)


def define_pmtg_regions(density_path, min_subjects):
    """Return a bilateral mask of the pMTG clusters in a thresholded density map.

    Also saves the cluster map and a pMTG label map (1 = left, 2 = right).
    """
    cluster_path = density_path.with_name(density_path.name.replace("_density", "_clusters"))
    pmtg_path = density_path.with_name(density_path.name.replace("_density", "_pmtg"))
    # Workbench keeps values above the threshold, so this keeps counts >= min_subjects
    find_clusters(density_path, min_subjects - 1, GROUP_MIN_CLUSTER_AREA_MM2, cluster_path)
    cluster_labels = load_cifti_vector(cluster_path)

    pmtg_labels = np.zeros(cluster_labels.shape)
    for region_value, hemisphere in [(1, "left"), (2, "right")]:
        hemisphere_slice = cortex[hemisphere]["slice"]
        hemisphere_labels = cluster_labels[hemisphere_slice]
        distances = {
            label: np.linalg.norm(
                centroid(cluster_labels == label, hemisphere) - PMTG_CENTERS[hemisphere]
            )
            for label in np.unique(hemisphere_labels[hemisphere_labels > 0])
        }
        nearest = min(distances, key=distances.get)
        pmtg_labels[hemisphere_slice] = np.where(hemisphere_labels == nearest, region_value, 0)

    save_dscalar(pmtg_labels, pmtg_path)
    return pmtg_labels > 0


full_pmtg = define_pmtg_regions(full_density_paths[10], min_subjects=5)

pd.DataFrame.from_dict(
    {hemisphere: centroid(full_pmtg, hemisphere) for hemisphere in ["left", "right"]},
    orient="index",
    columns=["x", "y", "z"],
    dtype=float,
).round(1)

,x,y,z
left,-57.9,-53.6,2.5
right,58.0,-48.6,4.6


## Diagnosis Exclusion

pMTG regions are redefined from the seven participants without a neurodevelopmental diagnosis (variant in at least 4 of 7) and compared with the full-cohort regions.

In [16]:
def dice(mask_a, mask_b):
    return 2 * np.sum(mask_a & mask_b) / (np.sum(mask_a) + np.sum(mask_b))


td_density_path = make_density_map(TD_SUBJECTS, 10, "td_only_bottom10")
td_pmtg = define_pmtg_regions(td_density_path, min_subjects=4)

diagnosis_exclusion_dice = dice(full_pmtg, td_pmtg)
print(f"Dice, diagnosis-excluded vs. full-cohort pMTG: {diagnosis_exclusion_dice:.2f}")

Dice, diagnosis-excluded vs. full-cohort pMTG: 0.90


## Jackknife Sensitivity

pMTG regions are redefined 10 times, each time leaving out one participant (variant in at least 4 of the remaining 9). Each jackknife region is first compared with the full-cohort region.

In [17]:
jackknife_pmtg = {}
for omitted in COHORT_SUBJECTS:
    retained = [subject for subject in COHORT_SUBJECTS if subject != omitted]
    density_path = make_density_map(retained, 10, f"jackknife_without_{omitted}_bottom10")
    jackknife_pmtg[omitted] = define_pmtg_regions(density_path, min_subjects=4)

jackknife_dice = pd.Series(
    {omitted: dice(full_pmtg, pmtg) for omitted, pmtg in jackknife_pmtg.items()},
    name="dice_vs_full_cohort",
).rename_axis("omitted_subject")
jackknife_dice.to_csv(OUTPUT_DIR / "jackknife_leave_one_out_bilateral_dice.csv")

display(jackknife_dice.round(2).to_frame())
print(f"Mean Dice = {jackknife_dice.mean():.2f} (SD = {jackknife_dice.std():.2f})")

,dice_vs_full_cohort
omitted_subject,
MSCPI05,0.87
MSCPI07,0.62
MSCPI08,0.76
MSCPI10,0.61
MSCPI12,0.81
MSCPI14,0.58
MSCPI15,0.88
MSCPI17,0.72
MSCPI18,0.67


Mean Dice = 0.73 (SD = 0.11)


Every pair of jackknife regions is then compared. Rows and columns name the participant left out of each jackknife region.

In [18]:
pairwise_dice = pd.DataFrame(np.nan, index=COHORT_SUBJECTS, columns=COHORT_SUBJECTS)
for first, second in combinations(COHORT_SUBJECTS, 2):
    pair_dice = dice(jackknife_pmtg[first], jackknife_pmtg[second])
    pairwise_dice.loc[first, second] = pair_dice
    pairwise_dice.loc[second, first] = pair_dice
pairwise_dice.to_csv(OUTPUT_DIR / "jackknife_pairwise_bilateral_pmtg_dice.csv")

min_pairwise_dice = np.nanmin(pairwise_dice.to_numpy())
display(pairwise_dice.round(2))
print(f"Minimum pairwise Dice = {min_pairwise_dice:.2f}")

,MSCPI05,MSCPI07,MSCPI08,MSCPI10,MSCPI12,MSCPI14,MSCPI15,MSCPI17,MSCPI18,MSCPI19
MSCPI05,NaN,0.69,0.80,0.70,0.79,0.69,0.84,0.65,0.71,0.81
MSCPI07,0.69,NaN,0.78,0.93,0.67,0.95,0.61,0.88,0.89,0.68
MSCPI08,0.80,0.78,NaN,0.77,0.80,0.79,0.78,0.66,0.81,0.84
MSCPI10,0.70,0.93,0.77,NaN,0.73,0.96,0.69,0.83,0.84,0.74
MSCPI12,0.79,0.67,0.80,0.73,NaN,0.74,0.89,0.71,0.76,0.87
MSCPI14,0.69,0.95,0.79,0.96,0.74,NaN,0.68,0.83,0.89,0.76
MSCPI15,0.84,0.61,0.78,0.69,0.89,0.68,NaN,0.66,0.69,0.90
MSCPI17,0.65,0.88,0.66,0.83,0.71,0.83,0.66,NaN,0.81,0.61
MSCPI18,0.71,0.89,0.81,0.84,0.76,0.89,0.69,0.81,NaN,0.77
MSCPI19,0.81,0.68,0.84,0.74,0.87,0.76,0.90,0.61,0.77,NaN


Minimum pairwise Dice = 0.61


## Summary

In [19]:
summary = pd.DataFrame(
    [
        ["Diagnosis-excluded vs. full cohort", "Dice", diagnosis_exclusion_dice],
        ["Jackknife vs. full cohort", "Mean Dice", jackknife_dice.mean()],
        ["Jackknife vs. full cohort", "SD of Dice", jackknife_dice.std()],
        ["Between jackknife regions", "Minimum Dice", min_pairwise_dice],
    ],
    columns=["comparison", "statistic", "value"],
)
summary.to_csv(OUTPUT_DIR / "pmtg_dice_summary.csv", index=False)
summary.round(2)

,comparison,statistic,value
0,Diagnosis-excluded vs. full cohort,Dice,0.90
1,Jackknife vs. full cohort,Mean Dice,0.73
2,Jackknife vs. full cohort,SD of Dice,0.11
3,Between jackknife regions,Minimum Dice,0.61
